# 强化学习结果描述性统计   

强化学习的学习曲线：  

由于存在多个agent，每个agent在每个窗口有多次训练，因此，学习曲线的绘制如下:  
1.为每一个agent绘制学习曲线，横轴为窗口，纵轴为该窗口所有奖励的均值   
2.求每个agent，每个窗口平均奖励的均值，作为该窗口的平均奖励，绘制平均学习曲线     
   


## 导入库

In [1]:
import json  
import math
import os
import re
import polars as pl
import plotly  
from plotly import express as px
from plotly.subplots import make_subplots
import plotly.graph_objects as go
from pathlib import Path
import warnings

## 超参数

In [2]:
TASK_ID_PREFIX = 'baseline1'  # 任务id前缀
RESULTS_BASE_DIR = '/home/frank/files/programs/GraduationThesis/result' # 基本数据路径
SAVE_BASE_DIR = f'/home/frank/files/programs/GraduationThesis/empirical/{TASK_ID_PREFIX}' # 保存基本路径
SAVE_BASELINE_REG_DIR = SAVE_BASE_DIR + '/baseline_reg'
SAVE = True # 是否保存数据

# 是否绘制单个Node奖励图像
SINGLE_NODE_FIG = False

# 单个Node子图配置
NODE_SUB_FIG_CONFIG = {
    'width': 2, 
    'height': 8,  
} # 单个Node所有证券的学习曲线构成子图，子图配置，长2宽8  

# 绘图的起始年份 -- 剔除最早的数据（由于波动）  
START_YEAR = 2012

# 平滑窗口
SMOOTH_WINDOW = 12


## 读取数据
数据的保存路径为：    

```
DATA_BASE_DIR  
|- TASK_ID1
|   |- Node1
|   |   |- performance_and_record_0.json   
|   |   |- performance_and_record_1.json
|   |   |- ...
|   |- Node2
|   |   |- performance_and_record_0.json
|   |   |- performance_and_record_1.json
|   |   |- ...
|   |- ...
|- TASK_ID2
|   |- Node1
|   |   |- performance_and_record_0.json
|   |   |- performance_and_record_1.json
|   |   |- ...
```  

其中，task_id的构成为 时间_中缀_uuid，例如：20260214_1924_lstm_short_66ab0230-9c11-4dc5-90e4-feb4b2c2fa57  
指定中缀，选取所有中缀一样的task_id，得到所有node的路径  

In [3]:
# 列出task_id下的所有no
pattern = re.compile(r'\d{8}_\d{4}_' + TASK_ID_PREFIX + r'_[a-z0-9\-]+')
matching_task_ids = [task_id for task_id in os.listdir(RESULTS_BASE_DIR) if pattern.match(task_id)] # 匹配中缀 
print(matching_task_ids)

# 获取其下所有node的路径
node_paths = [] # 所有该task_id中缀的node路径
for task_id in matching_task_ids:
    node_paths.extend(
        os.path.join(
            RESULTS_BASE_DIR, task_id, 
            node_path
        )
        for node_path in os.listdir(os.path.join(RESULTS_BASE_DIR, task_id))
    ) 

node_paths


['20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298', '20260301_1449_baseline1_ddf9ed20-a2ff-44cb-aebd-6b1d61f75c58', '20260225_1816_baseline1_1f77a8c2-c159-4769-ab53-4395774a51c7', '20260226_0941_baseline1_72e0fc17-d099-4509-b2ad-99f00af0ae55', '20260227_1115_baseline1_55a04a2a-9b27-487d-8354-7ba853498911', '20260227_2043_baseline1_108d0e4e-6715-412d-8c03-b7f4dcf28d7c', '20260228_0814_baseline1_26f7a6b7-012b-43de-a0d6-3a17fbfd7548', '20260226_1214_baseline1_1bfabc20-8ed2-4394-99b2-556b01929c48', '20260228_1730_baseline1_e16700c0-947d-4499-9506-12c045fa7747', '20260226_2036_baseline1_2c260f6a-dc21-4820-8e5a-309be7bd4344', '20260225_2335_baseline1_e7b3974a-00fe-4379-b701-e4c00dd4649f', '20260227_1516_baseline1_778167b9-99d7-42b7-95a8-cafdb64ff35c', '20260302_2057_baseline1_2ed5d96d-960e-4f11-af9a-ec2028c80b64', '20260302_0828_baseline1_3fb87bbb-bc56-4c1c-b9b3-ff61432998b2']


['/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014488_6824',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772031879_25181',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772031891_9646',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772031901_11800',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014501_6831',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014595_27084',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014583_23544',
 '/home/frank/files/programs/Graduati

In [4]:
# 获取其下所有performance_and_record_*.jsonl文件的路径
jsonl_paths = []
for node_path in node_paths:
    all_files = os.listdir(node_path)
    perf_and_rewa_jsonl_files = [file for file in all_files if file.endswith('.jsonl') and 'performance_and_reward_' in file]
    jsonl_paths.extend(
        os.path.join(node_path, file)
        for file in perf_and_rewa_jsonl_files
    )
jsonl_paths

['/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014488_6824/performance_and_reward_2.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014488_6824/performance_and_reward_3.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014488_6824/performance_and_reward_1.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772014488_6824/performance_and_reward_4.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772031879_25181/performance_and_reward_2.jsonl',
 '/home/frank/files/programs/GraduationThesis/result/20260227_2337_baseline1_feb6e541-b369-4244-bd89-500fd04a0298/node_1772031879_25181/performance_and_reward_

先用scan获取所有的jsonl，并且添加一个node字段，为lit(node_name)，node_name是路径的倒数第二个字符串    
使用polars的concat方法，合并所有json，字段由performance_and_record约定好  
展示测试数据  

In [5]:
lazy_frames = [pl.scan_ndjson(f).with_columns(pl.lit(Path(f).parent.stem).alias('node')) for f in jsonl_paths]
lf = pl.concat(lazy_frames)   # 得到 LazyFrame
lf.head().collect()

year,month,portfolio,data,node
i64,i64,list[str],struct[4],str
2022,1,"[""688071""]","{[1.0, 0.0],[-0.055, -0.0998, … -0.0],[0.0668, -9.9794, … -0.8893],-0.0717}","""node_1772014488_6824"""
2022,1,"[""301087""]","{[1.0, 0.0],[-0.0623, -0.0035, … -0.0],[0.0151, -0.3536, … -0.8893],-0.0623}","""node_1772014488_6824"""
2022,1,"[""603171""]","{[1.0, 0.0],[0.1262, -0.0631, … -0.0],[1.3609, -6.3076, … -0.8893],0.1195}","""node_1772014488_6824"""
2022,1,"[""688257""]","{[1.0, 0.0],[-0.1147, -0.0089, … -0.0],[-0.3593, -0.885, … -0.8893],-0.1148}","""node_1772014488_6824"""
2022,1,"[""688192""]","{[1.0, 0.0],[-0.0116, -0.0593, … -0.0],[0.3768, -5.9307, … -0.8893],-0.0175}","""node_1772014488_6824"""


## 处理数据  
加载的lf，其结构如上所示，需要进行解析   

>- 1.基线回归为单证券，因此，需要验证portfolio中列表长度，如果有长度大于1的列表，需要警告，并且取[0]  
>- 2.提取year,month,portfolio,reward  

In [6]:
# 定义字段
perf_fields = ["return", "neg_vol", "sharp", "neg_maxdrawdown", "diversification"]

# 1. 在 LazyFrame 上完成 unnest，再 collect（单次扫描，避免先 collect 再在 Python 里 unnest）
lf_unnested = lf.with_columns(pl.col("data").struct.unnest()).drop("data")
df_flat = lf_unnested.collect()  # 需要子集时可改为 lf_unnested.head(n).collect()

# 2. portfolio：检查是否有多证券，有则告警并只保留 [0]
if (df_flat["portfolio"].list.len() > 1).any():
    warnings.warn("发现 portfolio 列表长度 > 1（多证券），已取 [0] 作为单证券基线。")
df_flat = df_flat.with_columns(pl.col("portfolio").list.get(0).alias("portfolio"))

# 3.获取字段
df_flat = df_flat.select(
    pl.col(
        [
            'year',
            'month',
            'portfolio',
            'node', 
            'reward',
        ]
    )
)

In [7]:
df_flat.head()

year,month,portfolio,node,reward
i64,i64,str,str,f64
2022,1,"""688071""","""node_1772014488_6824""",-0.0717
2022,1,"""301087""","""node_1772014488_6824""",-0.0623
2022,1,"""603171""","""node_1772014488_6824""",0.1195
2022,1,"""688257""","""node_1772014488_6824""",-0.1148
2022,1,"""688192""","""node_1772014488_6824""",-0.0175


In [8]:
# 转为lf，方便后续处理
lf_flat = df_flat.lazy()

## 强化学习奖励图像  

### 单个node奖励 （可选）    
选取一个node，为每一个证券组合绘制一个曲线，横轴为年月，纵轴为reward     



In [9]:
if SINGLE_NODE_FIG:
    sub_figs = []
    for node in lf_flat.select(pl.col('node')).unique().collect().to_series().to_list():
        # 筛选单个节点
        filtered_lf = lf_flat.filter(pl.col('node') == node)
        
        # 生成date
        filtered_lf = filtered_lf.with_columns(pl.date(pl.col('year'), pl.col('month'), 1).alias('date'))
        
        # 筛选起始年之后的年份  
        filtered_lf = filtered_lf.filter(pl.col('date') >= pl.date(START_YEAR, 1, 1))
        
        # 缩尾
        filtered_lf = filtered_lf.with_columns(pl.when(pl.col('reward') < pl.col('reward').quantile(0.99)).then(pl.col('reward')).otherwise(pl.col('reward').quantile(0.99)).alias('reward'))
        filtered_lf = filtered_lf.with_columns(pl.when(pl.col('reward') > pl.col('reward').quantile(0.01)).then(pl.col('reward')).otherwise(pl.col('reward').quantile(0.01)).alias('reward'))
        filtered_lf = filtered_lf.with_columns(pl.col('reward').fill_null(strategy='forward'))

        # 平滑
        filtered_lf = filtered_lf.with_columns(pl.col('reward').rolling_mean(window_size=SMOOTH_WINDOW).alias('reward'))

        # 排序
        filtered_lf = filtered_lf.sort(['date','portfolio'])

        # 绘制
        sub_fig = px.line(filtered_lf.collect().to_pandas(), x='date', y='reward', color='portfolio', title=f'{node} 各资产组合奖励曲线')
        sub_figs.append(sub_fig)

    # 按 NODE_SUB_FIG_CONFIG 将每 width*height 个子图合并为一张图

    w = NODE_SUB_FIG_CONFIG['width']
    h = NODE_SUB_FIG_CONFIG['height']
    chunk_size = w * h

    for start in range(0, len(sub_figs), chunk_size):
        fig_count = 0
        batch = sub_figs[start:start + chunk_size]
        titles = [(batch[i].layout.title.text if i < len(batch) else '') for i in range(chunk_size)]
        fig_combined = make_subplots(rows=h, cols=w, subplot_titles=titles, vertical_spacing=0.06, horizontal_spacing=0.05)
        for i, fig in enumerate(batch):
            row, col = i // w + 1, i % w + 1
            for trace in fig.data:
                fig_combined.add_trace(trace, row=row, col=col)
        fig_combined.update_layout(title_text='各 Node 学习曲线', height=280 * h, showlegend=False)
        fig_combined.show()
        fig_count += 1
        if SAVE:
            dir_path = Path(SAVE_BASELINE_REG_DIR)
            fig_combined.write_image(os.path.join(dir_path, f'基准回归-单节点学习曲线{fig_count}.png'))

### Node平均奖励  
对lf_flat，按照year,month求reward的平均值，作为该年月的平均奖励，绘制平均奖励曲线   

需要剔除异常值，主要是缩尾（0.01, 0.99）和前向填充 

绘制均值和置信区间曲线(95%)

In [10]:
# 添加date字段
lf_flat = lf_flat.with_columns(pl.date(pl.col('year'), pl.col('month'), 1).alias('date'))

# 筛选起始年之后的年份  
lf_flat = lf_flat.filter(pl.col('date') >= pl.date(START_YEAR, 1, 1))

# 缩尾
lf_flat = lf_flat.with_columns(pl.when(pl.col('reward') < pl.col('reward').quantile(0.01)).then(pl.col('reward').quantile(0.01)).otherwise(pl.col('reward')).alias('reward'))
lf_flat = lf_flat.with_columns(pl.when(pl.col('reward') > pl.col('reward').quantile(0.99)).then(pl.col('reward').quantile(0.99)).otherwise(pl.col('reward')).alias('reward'))

# 计算均值
mean_reward_df = lf_flat.group_by(['date']).agg(pl.col('reward').mean())

# 平滑
# mean_reward_df = mean_reward_df.with_columns(pl.col('reward').rolling_mean(window_size=SMOOTH_WINDOW).alias('reward'))

# 计算置信区间
mean_reward_df = mean_reward_df.with_columns((pl.col('reward').std() / pl.col('reward').count().sqrt()).alias('std_error'))
mean_reward_df = mean_reward_df.with_columns((pl.col('reward') - 1.96 * pl.col('std_error')).alias('lower_ci'))
mean_reward_df = mean_reward_df.with_columns((pl.col('reward') + 1.96 * pl.col('std_error')).alias('upper_ci'))

# 排序+填充
mean_reward_df = mean_reward_df.sort(['date'])
mean_reward_df = mean_reward_df.with_columns(pl.col('reward').fill_null(strategy='forward'))
plot_df = mean_reward_df.collect().to_pandas()

In [11]:
# 绘制平均奖励曲线
mean_line_fig = px.line(plot_df, x='date', y='reward', title='平均奖励曲线')

# 先加下界线，再加上界线并填满与下界之间的区域
mean_line_fig.add_trace(go.Scatter(
    x=plot_df['date'], y=plot_df['lower_ci'], mode='lines',
    line=dict(color='rgba(100,100,100,0.6)', dash='dash'),
    name='95% CI 下界', showlegend=True,
))
mean_line_fig.add_trace(go.Scatter(
    x=plot_df['date'], y=plot_df['upper_ci'], mode='lines',
    fill='tonexty', fillcolor='rgba(100,149,237,0.25)',
    line=dict(color='rgba(100,100,100,0.6)', dash='dash'),
    name='95% CI 上界', showlegend=True,
))

mean_line_fig.show()

In [12]:
if SAVE:
    dir_path = Path(SAVE_BASELINE_REG_DIR)
    mean_line_fig.write_image(os.path.join(dir_path, f'基准回归-平均奖励曲线.png'))